In [1]:
# compute old FR average and new FR average network and nFR value 
# param nedge
import os
import sys  
import pandas as pd
import hgt_gcn
# import util


hgt_file = '../../hgt_abd_metadata/ERP010700.HGT.v2.short.csv'
# metadata_file = '../../hgt_abd_metadata/ERP010700.metadata.v2.tsv'
abd_file = '../../hgt_abd_metadata/ERP010700.merged.short10.tsv'
sp_file = '../../hgt_abd_metadata/genome_species.tsv'
db_dir = '../../HGT_demo_file/DB.genome_annotation'
gcn_file = '../../GCN_s.tsv'
d_file = '../../sp_d.tsv'
top_n = 100

hgt_df = pd.read_csv(hgt_file, index_col=None, header=0)
#metadata_df = pd.read_csv(metadata_file, index_col=None, header=0, sep='\t')
abd_df = pd.read_csv(abd_file, index_col=0, header=0, sep='\t')
sp_df = pd.read_csv(sp_file, index_col=0, header=0, sep='\t')
gcn_df = pd.read_csv(gcn_file, index_col=0, header=0, sep='\t')
sp_d = pd.read_csv(d_file, index_col=0, header=0, sep='\t')

#group = util.metadata2gf(metadata_df,groupid)
#if not util.check_valid(group, abd_df):
#    exit(2)


In [2]:
abd_df = hgt_gcn.multi_sample_normalize(abd_df)
genome_ko = hgt_gcn.ko_df(hgt_df, db_dir)


In [3]:
sp_ko_df = hgt_gcn.hgt2sp_ko(sp_df, genome_ko)

In [4]:
# multi sample test
nfr_result_df = pd.DataFrame(columns=['sample', 'nFR', 'adj_nFR'])
sum_fr_net = pd.DataFrame()
sum_adj_fr_net = pd.DataFrame()
for sname in list(abd_df.columns):
    nfr_result_df.loc[sname, 'sample'] = sname
    part_abd_df = abd_df[sname]
    part_abd_df = part_abd_df[part_abd_df > 0]
    tmp_abd = list(part_abd_df.index)
    part_df = sp_ko_df[sp_ko_df['sample'] == sname][['sp1', 'sp2', 'ko', 'num']]
    common_sp = list(set(tmp_abd).intersection(set(gcn_df.index)))
    tmp_d = sp_d.loc[common_sp, common_sp]
    # original fr
    nfr_value, fr_df, profile = hgt_gcn.nfr(tmp_d, abd_df, sname)
    nfr_result_df.loc[sname, 'nFR'] = nfr_value
    # align and add to sum nfr net
    sum_fr_net = hgt_gcn.net_sum(sum_fr_net, fr_df)
    if len(part_df)>0:
        new_gcn_df, effect_list = hgt_gcn.hgt_adjust_gcn(gcn_df, part_df)
        effect_list = list(set(effect_list).intersection(set(common_sp)))
        tmp_gcn = gcn_df.T[common_sp]
        if len(effect_list) < 80:
            new_d = hgt_gcn.adjust_d(tmp_d, tmp_gcn, effect_list)
        else:
            new_d = hgt_gcn.make_d(new_gcn_df.loc[common_sp,])
    
        nfr_value, fr_df, profile = hgt_gcn.nfr(new_d, abd_df, sname)
        nfr_result_df.loc[sname, 'adj_nFR'] = nfr_value
        sum_adj_fr_net = hgt_gcn.net_sum(sum_adj_fr_net, fr_df)
    else:
        nfr_result_df.loc[sname, 'adj_nFR'] = nfr_value
        sum_adj_fr_net = hgt_gcn.net_sum(sum_adj_fr_net, fr_df)
avg_fr_net = sum_fr_net / len(abd_df.columns)
avg_adj_fr_net = sum_adj_fr_net / len(abd_df.columns)
    


In [9]:
# top n 
avg_fr_output = hgt_gcn.output_fr_net(avg_fr_net, top_n)[0]
avg_adj_fr_output = hgt_gcn.output_fr_net(avg_adj_fr_net, top_n)[0]


In [10]:
avg_fr_output.columns = ['species1', 'species2', 'weight']

In [11]:
avg_fr_output

,species1,species2,weight
8698622,s__RC9_sp000433355,s__RC9_sp900544195,0.001653
981857,s__Bacteroides_uniformis,s__Phocaeicola_dorei,0.001515
399607,s__Alistipes_sp000434235,s__Phocaeicola_dorei,0.001355
8688339,s__RC9_sp000431015,s__RC9_sp000433355,0.000893
848282,s__Bacteroides_cellulosilyticus,s__Phocaeicola_dorei,0.000867
...,...,...,...
370016,s__Alistipes_communis,s__Alistipes_sp000434235,0.000114
981864,s__Bacteroides_uniformis,s__Phocaeicola_sartorii,0.000113
961307,s__Bacteroides_sp902362375,s__Phocaeicola_dorei,0.000113
920207,s__Bacteroides_sp002491635,s__Phocaeicola_dorei,0.000112
